# 03 — Decoding the *Dimension*: Valence vs Arousal

**Purpose:** Build a binary decoder whose two classes are not "positive vs
negative," but **valence trials vs arousal trials**. This is where the idea that
"the label *is* the hypothesis" becomes concrete: you keep all 64 images and
simply **relabel** them.

---

## Notebook overview

1. Load `sub-097_valence_vs_arousal_catalog.csv` (all 64 trials, labels =
   `valence` / `arousal`).
2. Build `imgs`, `y`, `groups`.
3. Set up a per-classifier **run folder** for this decoder's outputs.
4. Reuse the same decoder pipeline.
5. Fit, score against chance (still 50%), and interpret what a high or low score
   would *mean* scientifically.

## What you are learning

- How **relabeling** the same images creates a genuinely different question.
- Why "what does this decoder's accuracy tell me?" depends entirely on the labels.
- That cross-validation grouping (`groups`) still matters no matter the labels.
- How to organise outputs into a timestamped run folder (provenance).

## Why this matters scientifically

The first two decoders asked "within a dimension, can we tell positive from
negative?" This one asks "can we tell *which dimension was being modulated at
all*?" Those are different claims about how affect is organised in the brain.
Above-chance decoding here would suggest valence-engaging and arousal-engaging
trials evoke distinguishable whole-brain states — independent of their positive
/negative sign.

---
## Section 1 — Relabeling: the Conceptual Core

In notebooks 01–02, `label` was just a copy of `condition`. Here `label` is a
*derived* category that **collapses** four conditions into two:

| condition | label (`dimension`) |
|---|---|
| `pos_val` | valence |
| `neg_val` | valence |
| `pos_aro` | arousal |
| `neg_aro` | arousal |

You already created this `dimension` label in notebook 00 and saved it to
`sub-097_valence_vs_arousal_catalog.csv`. The image files are **byte-for-byte the
same** as in the other decoders — only the column you call `y` has changed. That
is the entire difference, and it changes the scientific question completely.

**TODO (no code — answer below):**
- [ ] Two decoders read the *same* beta maps but report very different accuracies.
      How is that possible, given the images are identical?

*Your answer:*

> (write here)

---
## Section 2 — Load the Table and Build `imgs`, `y`, `groups`

Same mechanics as before; the table just has 64 rows now and `label` is the
dimension. The file you saved from notebook 00 is
`sub-097_valence_vs_arousal_catalog.csv`.

**TODO:**
- [ ] Read `sub-097_valence_vs_arousal_catalog.csv` into `table`.
- [ ] Confirm `table["label"].value_counts()` is 32 `valence` / 32 `arousal`.
- [ ] Build `imgs`, `y`, `groups`; assert equal lengths.

In [15]:
from pathlib import Path
import pandas as pd
import numpy as np

subject = "sub-097"
project_dir  = Path(r"C:\ManzaRotation")
decoding_dir = project_dir / "Analysis" / "outputs" / subject / "decoding"
tables_dir   = decoding_dir / "tables"

# TODO: read tables_dir / f"{subject}_valence_vs_arousal_catalog.csv" into `table`
table = pd.read_csv(tables_dir / f"{subject}_valence_vs_arousal_catalog.csv")

# TODO: check 32 / 32 balance with table["label"].value_counts()
table["label"].value_counts()
# TODO: build imgs, y, groups from the path / label / groups columns (.tolist())
imgs = table["path"].tolist()
y = table["label"].tolist()
groups = table["groups"].tolist()
# TODO: assert len(imgs) == len(y) == len(groups)
assert(len(imgs)==len(y)==len(groups))

---
## Section 3 — A Balance Subtlety Worth Noticing

There is a quiet trap in this design. Each *run* contains all four conditions, so
each run is also 50/50 valence/arousal — good. But notice that "valence" pools
`pos_val` **and** `neg_val`. If the positive/negative split were uneven across
runs, the decoder might pick up something other than the dimension itself.

For `sub-097` the design is balanced, so this is fine — but the habit of asking
*"what else could my decoder be latching onto?"* is exactly the skepticism good
MVPA requires. The `groups` / leave-one-run-out scheme guards against the most
obvious confound (run identity); confounds *within* a label are subtler.

**TODO:**
- [ ] Print `table.groupby(["groups", "label"]).size()` and confirm each run is
      16 valence / 16 arousal.

In [16]:
table.groupby(["groups", "label"]).size() #to inspect per-run balance

groups     label  
modulate1  arousal    16
           valence    16
modulate2  arousal    16
           valence    16
dtype: int64

---
## Section 4 — Set Up a Run Folder for This Classifier

Before fitting, create a place to put this decoder's outputs. Reusing a single
flat `figures/` folder for every decoder would let runs overwrite each other and
lose track of *which settings produced which result*. Instead — exactly like the
beta-series `runs/` folders and the finished decoders in notebooks 01 and 02 —
give **each decoding question its own top-level folder**, and **each run its own
timestamped subfolder**:

```
decoding/
  valence_vs_arousal_decoding/        <- one folder per decoding question
    runs/
      2026-06-25_1530_valence_vs_arousal_svc/   <- one folder per run
        figures/      # weight maps, etc.
        scores/       # the results CSV
        params.json   # exactly which settings produced this run
```

Why bother? **Provenance.** Months from now, a stray PNG means nothing unless you
can trace it back to the mask, screening percentile, and CV scheme that made it.
The `params.json` is your receipt.

The building blocks you need:

```python
from datetime import datetime
import json

timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
run_dir   = decoder_dir / "runs" / f"{timestamp}_{analysis_name}"
figures_dir = run_dir / "figures"
results_dir = run_dir / "scores"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)
```

**TODO (fill in the cell below):**
- [ ] Define `decoder_dir = decoding_dir / "valence_vs_arousal_decoding"`.
- [ ] Set `analysis_name = "valence_vs_arousal_svc"`.
- [ ] Build `timestamp`, `run_dir`, `figures_dir`, `results_dir` and create them.
- [ ] (Later, in Section 5) write a `params.json` into `run_dir` and save your
      scores CSV into `results_dir`.

In [ ]:
from datetime import datetime
import json

# A "run folder" keeps everything from one fitting run together — figures,
# scores, and a params.json — under a timestamped subfolder. This mirrors the
# runs/ convention from the beta-series notebooks and the finished valence /
# arousal decoders (notebooks 01 and 02). One TOP folder per decoding question;
# one timestamped subfolder per time you run it.

# TODO: name a folder for THIS classifier (the valence-vs-arousal question)
decoder_dir = decoding_dir / "valence_vs_arousal_decoding"
decoder_dir.mkdir(exist_ok=True)
runs_dir = decoder_dir / "runs"
runs_dir.mkdir(exist_ok=True)
# TODO: give this analysis a short name string, e.g. "valence_vs_arousal_svc"
analysis_name = "valence_vs_arousal_svc"

run_number = len(list(runs_dir.glob("run_*")))+1
# TODO: build a timestamp string, e.g. datetime.now().strftime("%Y-%m-%d_%H%M")
timestamp = datetime.now().strftime("%Y-%m-%d_%H%M")
run_dir = runs_dir / f"run_{run_number:03d}_{timestamp}_{analysis_name}"

results_dir = run_dir / "scores"
figures_dir = run_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

print(run_dir)


# TODO: build figures_dir = run_dir / "figures"  and  results_dir = run_dir / "scores"
# TODO: create BOTH with .mkdir(parents=True, exist_ok=True)
# (creating figures_dir/results_dir also creates run_dir for you)

# print(run_dir)   # uncomment once defined, to confirm where outputs will land

C:\ManzaRotation\Analysis\outputs\sub-097\decoding\valence_vs_arousal_decoding\runs\run_001_2026-06-26_1447_valence_vs_arousal_svc


: 

---
## Section 5 — Reuse the Pipeline, Fit, and Score

The decoder setup is unchanged from notebook 01. Chance is still **50%** (32 vs
32). Save this run's outputs into the `figures_dir` / `results_dir` you just made,
and drop a `params.json` into `run_dir` (look at notebook 01's fit cell for the
finished version of this pattern).

**TODO:**
- [ ] Point `mask_path` at the brain mask.
- [ ] Build `cv = LeaveOneGroupOut()` and the same `Decoder` configuration.
- [ ] `decoder.fit(imgs, y, groups=groups)`.
- [ ] Compute and print the mean cross-validated accuracy vs 50%.
- [ ] Save the scores to `results_dir`, any weight maps to `figures_dir`, and
      write `params.json` into `run_dir`.

In [ ]:
from nilearn.decoding import Decoder
from sklearn.model_selection import LeaveOneGroupOut

func_deriv_dir = project_dir / "Derivatives" / subject / "func"
# TODO: mask_path = ...
mask_path = func_deriv_dir
cv = LeaveOneGroupOut()
decoder = Decoder(estimator="svc", mask=mask_path, standardize="zscore_sample",
#                         screening_percentile=..., scoring="accuracy", cv=cv)
# TODO: decoder.fit(imgs, y, groups=groups)
# TODO: print mean accuracy vs chance (50%)

---
## Section 6 — Interpret: What Would a High vs Low Score Mean?

Spell out the science *before* trusting the number:

- **Clearly above 50%:** valence-modulating and arousal-modulating trials evoke
  distinguishable whole-brain patterns — the brain "knows" which dimension was in
  play, beyond the positive/negative sign.
- **Near 50%:** no evidence (in this subject, with this mask and pipeline) that
  the dimension is decodable. That is a real result too, not a failure — but
  remember absence of evidence isn't evidence of absence, especially with two
  folds.

**TODO:**
- [ ] Write your interpretation in the markdown cell below.
- [ ] Note one follow-up analysis that would strengthen whatever you conclude
      (e.g. permutation test for an empirical chance distribution, more subjects,
      an ROI mask).

## Summary & what comes next

You have now seen three binary questions built from the same images by changing
labels. Notebook 04 steps up to **multiclass** decoding — all four conditions at
once — where chance drops to 25% and a **confusion matrix** becomes the key tool
for understanding *which* conditions the decoder confuses.

*Your interpretation and proposed follow-up:*

> (write here)